In [44]:
import re

In [45]:
data = {"idx": "120", "commit_id": "60fe637bf0e4d7989e21e50f52526444765c63b4", "project": "qemu", "func_name": "qemu_rdma_registration_handle", "language": "cpp", "code": "static int qemu_rdma_registration_handle(QEMUFile *f, void *opaque,\n\n                                         uint64_t flags)\n\n{\n\n    RDMAControlHeader reg_resp = { .len = sizeof(RDMARegisterResult),\n\n                               .type = RDMA_CONTROL_REGISTER_RESULT,\n\n                               .repeat = 0,\n\n                             };\n\n    RDMAControlHeader unreg_resp = { .len = 0,\n\n                               .type = RDMA_CONTROL_UNREGISTER_FINISHED,\n\n                               .repeat = 0,\n\n                             };\n\n    RDMAControlHeader blocks = { .type = RDMA_CONTROL_RAM_BLOCKS_RESULT,\n\n                                 .repeat = 1 };\n\n    QEMUFileRDMA *rfile = opaque;\n\n    RDMAContext *rdma = rfile->rdma;\n\n    RDMALocalBlocks *local = &rdma->local_ram_blocks;\n\n    RDMAControlHeader head;\n\n    RDMARegister *reg, *registers;\n\n    RDMACompress *comp;\n\n    RDMARegisterResult *reg_result;\n\n    static RDMARegisterResult results[RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE];\n\n    RDMALocalBlock *block;\n\n    void *host_addr;\n\n    int ret = 0;\n\n    int idx = 0;\n\n    int count = 0;\n\n    int i = 0;\n\n\n\n    CHECK_ERROR_STATE();\n\n\n\n    do {\n\n        DDDPRINTF(\"Waiting for next request %\" PRIu64 \"...\\n\", flags);\n\n\n\n        ret = qemu_rdma_exchange_recv(rdma, &head, RDMA_CONTROL_NONE);\n\n\n\n        if (ret < 0) {\n\n            break;\n\n        }\n\n\n\n        if (head.repeat > RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE) {\n\n            fprintf(stderr, \"rdma: Too many requests in this message (%d).\"\n\n                            \"Bailing.\\n\", head.repeat);\n\n            ret = -EIO;\n\n            break;\n\n        }\n\n\n\n        switch (head.type) {\n\n        case RDMA_CONTROL_COMPRESS:\n\n            comp = (RDMACompress *) rdma->wr_data[idx].control_curr;\n\n            network_to_compress(comp);\n\n\n\n            DDPRINTF(\"Zapping zero chunk: %\" PRId64\n\n                    \" bytes, index %d, offset %\" PRId64 \"\\n\",\n\n                    comp->length, comp->block_idx, comp->offset);\n\n            block = &(rdma->local_ram_blocks.block[comp->block_idx]);\n\n\n\n            host_addr = block->local_host_addr +\n\n                            (comp->offset - block->offset);\n\n\n\n            ram_handle_compressed(host_addr, comp->value, comp->length);\n\n            break;\n\n\n\n        case RDMA_CONTROL_REGISTER_FINISHED:\n\n            DDDPRINTF(\"Current registrations complete.\\n\");\n\n            goto out;\n\n\n\n        case RDMA_CONTROL_RAM_BLOCKS_REQUEST:\n\n            DPRINTF(\"Initial setup info requested.\\n\");\n\n\n\n            if (rdma->pin_all) {\n\n                ret = qemu_rdma_reg_whole_ram_blocks(rdma);\n\n                if (ret) {\n\n                    fprintf(stderr, \"rdma migration: error dest \"\n\n                                    \"registering ram blocks!\\n\");\n\n                    goto out;\n\n                }\n\n            }\n\n\n\n            /*\n\n             * Dest uses this to prepare to transmit the RAMBlock descriptions\n\n             * to the source VM after connection setup.\n\n             * Both sides use the \"remote\" structure to communicate and update\n\n             * their \"local\" descriptions with what was sent.\n\n             */\n\n            for (i = 0; i < local->nb_blocks; i++) {\n\n                rdma->block[i].remote_host_addr =\n\n                    (uint64_t)(local->block[i].local_host_addr);\n\n\n\n                if (rdma->pin_all) {\n\n                    rdma->block[i].remote_rkey = local->block[i].mr->rkey;\n\n                }\n\n\n\n                rdma->block[i].offset = local->block[i].offset;\n\n                rdma->block[i].length = local->block[i].length;\n\n\n\n                remote_block_to_network(&rdma->block[i]);\n\n            }\n\n\n\n            blocks.len = rdma->local_ram_blocks.nb_blocks\n\n                                                * sizeof(RDMARemoteBlock);\n\n\n\n\n\n            ret = qemu_rdma_post_send_control(rdma,\n\n                                        (uint8_t *) rdma->block, &blocks);\n\n\n\n            if (ret < 0) {\n\n                fprintf(stderr, \"rdma migration: error sending remote info!\\n\");\n\n                goto out;\n\n            }\n\n\n\n            break;\n\n        case RDMA_CONTROL_REGISTER_REQUEST:\n\n            DDPRINTF(\"There are %d registration requests\\n\", head.repeat);\n\n\n\n            reg_resp.repeat = head.repeat;\n\n            registers = (RDMARegister *) rdma->wr_data[idx].control_curr;\n\n\n\n            for (count = 0; count < head.repeat; count++) {\n\n                uint64_t chunk;\n\n                uint8_t *chunk_start, *chunk_end;\n\n\n\n                reg = &registers[count];\n\n                network_to_register(reg);\n\n\n\n                reg_result = &results[count];\n\n\n\n                DDPRINTF(\"Registration request (%d): index %d, current_addr %\"\n\n                         PRIu64 \" chunks: %\" PRIu64 \"\\n\", count,\n\n                         reg->current_index, reg->key.current_addr, reg->chunks);\n\n\n\n                block = &(rdma->local_ram_blocks.block[reg->current_index]);\n\n                if (block->is_ram_block) {\n\n                    host_addr = (block->local_host_addr +\n\n                                (reg->key.current_addr - block->offset));\n\n                    chunk = ram_chunk_index(block->local_host_addr,\n\n                                            (uint8_t *) host_addr);\n\n                } else {\n\n                    chunk = reg->key.chunk;\n\n                    host_addr = block->local_host_addr +\n\n                        (reg->key.chunk * (1UL << RDMA_REG_CHUNK_SHIFT));\n\n                }\n\n                chunk_start = ram_chunk_start(block, chunk);\n\n                chunk_end = ram_chunk_end(block, chunk + reg->chunks);\n\n                if (qemu_rdma_register_and_get_keys(rdma, block,\n\n                            (uint8_t *)host_addr, NULL, &reg_result->rkey,\n\n                            chunk, chunk_start, chunk_end)) {\n\n                    fprintf(stderr, \"cannot get rkey!\\n\");\n\n                    ret = -EINVAL;\n\n                    goto out;\n\n                }\n\n\n\n                reg_result->host_addr = (uint64_t) block->local_host_addr;\n\n\n\n                DDPRINTF(\"Registered rkey for this request: %x\\n\",\n\n                                reg_result->rkey);\n\n\n\n                result_to_network(reg_result);\n\n            }\n\n\n\n            ret = qemu_rdma_post_send_control(rdma,\n\n                            (uint8_t *) results, &reg_resp);\n\n\n\n            if (ret < 0) {\n\n                fprintf(stderr, \"Failed to send control buffer!\\n\");\n\n                goto out;\n\n            }\n\n            break;\n\n        case RDMA_CONTROL_UNREGISTER_REQUEST:\n\n            DDPRINTF(\"There are %d unregistration requests\\n\", head.repeat);\n\n            unreg_resp.repeat = head.repeat;\n\n            registers = (RDMARegister *) rdma->wr_data[idx].control_curr;\n\n\n\n            for (count = 0; count < head.repeat; count++) {\n\n                reg = &registers[count];\n\n                network_to_register(reg);\n\n\n\n                DDPRINTF(\"Unregistration request (%d): \"\n\n                         \" index %d, chunk %\" PRIu64 \"\\n\",\n\n                         count, reg->current_index, reg->key.chunk);\n\n\n\n                block = &(rdma->local_ram_blocks.block[reg->current_index]);\n\n\n\n                ret = ibv_dereg_mr(block->pmr[reg->key.chunk]);\n\n                block->pmr[reg->key.chunk] = NULL;\n\n\n\n                if (ret != 0) {\n\n                    perror(\"rdma unregistration chunk failed\");\n\n                    ret = -ret;\n\n                    goto out;\n\n                }\n\n\n\n                rdma->total_registrations--;\n\n\n\n                DDPRINTF(\"Unregistered chunk %\" PRIu64 \" successfully.\\n\",\n\n                            reg->key.chunk);\n\n            }\n\n\n\n            ret = qemu_rdma_post_send_control(rdma, NULL, &unreg_resp);\n\n\n\n            if (ret < 0) {\n\n                fprintf(stderr, \"Failed to send control buffer!\\n\");\n\n                goto out;\n\n            }\n\n            break;\n\n        case RDMA_CONTROL_REGISTER_RESULT:\n\n            fprintf(stderr, \"Invalid RESULT message at dest.\\n\");\n\n            ret = -EIO;\n\n            goto out;\n\n        default:\n\n            fprintf(stderr, \"Unknown control message %s\\n\",\n\n                                control_desc[head.type]);\n\n            ret = -EIO;\n\n            goto out;\n\n        }\n\n    } while (1);\n\nout:\n\n    if (ret < 0) {\n\n        rdma->error_state = ret;\n\n    }\n\n    return ret;\n\n}\n", "code_tokens": ["static", "int", "qemu_rdma_registration_handle", "(", "QEMUFile", "*", "f", ",", "void", "*", "opaque", ",", "uint64_t", "flags", ")", "{", "RDMAControlHeader", "reg_resp", "=", "{", ".", "len", "=", "sizeof", "(", "RDMARegisterResult", ")", ",", ".", "type", "=", "RDMA_CONTROL_REGISTER_RESULT", ",", ".", "repeat", "=", "0", ",", "}", ";", "RDMAControlHeader", "unreg_resp", "=", "{", ".", "len", "=", "0", ",", ".", "type", "=", "RDMA_CONTROL_UNREGISTER_FINISHED", ",", ".", "repeat", "=", "0", ",", "}", ";", "RDMAControlHeader", "blocks", "=", "{", ".", "type", "=", "RDMA_CONTROL_RAM_BLOCKS_RESULT", ",", ".", "repeat", "=", "1", "}", ";", "QEMUFileRDMA", "*", "rfile", "=", "opaque", ";", "RDMAContext", "*", "rdma", "=", "rfile", "->", "rdma", ";", "RDMALocalBlocks", "*", "local", "=", "&", "rdma", "->", "local_ram_blocks", ";", "RDMAControlHeader", "head", ";", "RDMARegister", "*", "reg", ",", "*", "registers", ";", "RDMACompress", "*", "comp", ";", "RDMARegisterResult", "*", "reg_result", ";", "static", "RDMARegisterResult", "results", "[", "RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE", "]", ";", "RDMALocalBlock", "*", "block", ";", "void", "*", "host_addr", ";", "int", "ret", "=", "0", ";", "int", "idx", "=", "0", ";", "int", "count", "=", "0", ";", "int", "i", "=", "0", ";", "CHECK_ERROR_STATE", "(", ")", ";", "do", "{", "DDDPRINTF", "(", "\"", "Waiting for next request %", "\"", "PRIu64", "\"", "...", "\\n", "\"", ",", "flags", ")", ";", "ret", "=", "qemu_rdma_exchange_recv", "(", "rdma", ",", "&", "head", ",", "RDMA_CONTROL_NONE", ")", ";", "if", "(", "ret", "<", "0", ")", "{", "break", ";", "}", "if", "(", "head", ".", "repeat", ">", "RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE", ")", "{", "fprintf", "(", "stderr", ",", "\"", "rdma: Too many requests in this message (%d).", "\"", "\"", "Bailing.", "\\n", "\"", ",", "head", ".", "repeat", ")", ";", "ret", "=", "-", "EIO", ";", "break", ";", "}", "switch", "(", "head", ".", "type", ")", "{", "case", "RDMA_CONTROL_COMPRESS", ":", "comp", "=", "(", "RDMACompress", "*", ")", "rdma", "->", "wr_data", "[", "idx", "]", ".", "control_curr", ";", "network_to_compress", "(", "comp", ")", ";", "DDPRINTF", "(", "\"", "Zapping zero chunk: %", "\"", "PRId64", "\"", " bytes, index %d, offset %", "\"", "PRId64", "\"", "\\n", "\"", ",", "comp", "->", "length", ",", "comp", "->", "block_idx", ",", "comp", "->", "offset", ")", ";", "block", "=", "&", "(", "rdma", "->", "local_ram_blocks", ".", "block", "[", "comp", "->", "block_idx", "]", ")", ";", "host_addr", "=", "block", "->", "local_host_addr", "+", "(", "comp", "->", "offset", "-", "block", "->", "offset", ")", ";", "ram_handle_compressed", "(", "host_addr", ",", "comp", "->", "value", ",", "comp", "->", "length", ")", ";", "break", ";", "case", "RDMA_CONTROL_REGISTER_FINISHED", ":", "DDDPRINTF", "(", "\"", "Current registrations complete.", "\\n", "\"", ")", ";", "goto", "out", ";", "case", "RDMA_CONTROL_RAM_BLOCKS_REQUEST", ":", "DPRINTF", "(", "\"", "Initial setup info requested.", "\\n", "\"", ")", ";", "if", "(", "rdma", "->", "pin_all", ")", "{", "ret", "=", "qemu_rdma_reg_whole_ram_blocks", "(", "rdma", ")", ";", "if", "(", "ret", ")", "{", "fprintf", "(", "stderr", ",", "\"", "rdma migration: error dest ", "\"", "\"", "registering ram blocks!", "\\n", "\"", ")", ";", "goto", "out", ";", "}", "}", "for", "(", "i", "=", "0", ";", "i", "<", "local", "->", "nb_blocks", ";", "i", "++", ")", "{", "rdma", "->", "block", "[", "i", "]", ".", "remote_host_addr", "=", "(", "uint64_t", ")", "(", "local", "->", "block", "[", "i", "]", ".", "local_host_addr", ")", ";", "if", "(", "rdma", "->", "pin_all", ")", "{", "rdma", "->", "block", "[", "i", "]", ".", "remote_rkey", "=", "local", "->", "block", "[", "i", "]", ".", "mr", "->", "rkey", ";", "}", "rdma", "->", "block", "[", "i", "]", ".", "offset", "=", "local", "->", "block", "[", "i", "]", ".", "offset", ";", "rdma", "->", "block", "[", "i", "]", ".", "length", "=", "local", "->", "block", "[", "i", "]", ".", "length", ";", "remote_block_to_network", "(", "&", "rdma", "->", "block", "[", "i", "]", ")", ";", "}", "blocks", ".", "len", "=", "rdma", "->", "local_ram_blocks", ".", "nb_blocks", "*", "sizeof", "(", "RDMARemoteBlock", ")", ";", "ret", "=", "qemu_rdma_post_send_control", "(", "rdma", ",", "(", "uint8_t", "*", ")", "rdma", "->", "block", ",", "&", "blocks", ")", ";", "if", "(", "ret", "<", "0", ")", "{", "fprintf", "(", "stderr", ",", "\"", "rdma migration: error sending remote info!", "\\n", "\"", ")", ";", "goto", "out", ";", "}", "break", ";", "case", "RDMA_CONTROL_REGISTER_REQUEST", ":", "DDPRINTF", "(", "\"", "There are %d registration requests", "\\n", "\"", ",", "head", ".", "repeat", ")", ";", "reg_resp", ".", "repeat", "=", "head", ".", "repeat", ";", "registers", "=", "(", "RDMARegister", "*", ")", "rdma", "->", "wr_data", "[", "idx", "]", ".", "control_curr", ";", "for", "(", "count", "=", "0", ";", "count", "<", "head", ".", "repeat", ";", "count", "++", ")", "{", "uint64_t", "chunk", ";", "uint8_t", "*", "chunk_start", ",", "*", "chunk_end", ";", "reg", "=", "&", "registers", "[", "count", "]", ";", "network_to_register", "(", "reg", ")", ";", "reg_result", "=", "&", "results", "[", "count", "]", ";", "DDPRINTF", "(", "\"", "Registration request (%d): index %d, current_addr %", "\"", "PRIu64", "\"", " chunks: %", "\"", "PRIu64", "\"", "\\n", "\"", ",", "count", ",", "reg", "->", "current_index", ",", "reg", "->", "key", ".", "current_addr", ",", "reg", "->", "chunks", ")", ";", "block", "=", "&", "(", "rdma", "->", "local_ram_blocks", ".", "block", "[", "reg", "->", "current_index", "]", ")", ";", "if", "(", "block", "->", "is_ram_block", ")", "{", "host_addr", "=", "(", "block", "->", "local_host_addr", "+", "(", "reg", "->", "key", ".", "current_addr", "-", "block", "->", "offset", ")", ")", ";", "chunk", "=", "ram_chunk_index", "(", "block", "->", "local_host_addr", ",", "(", "uint8_t", "*", ")", "host_addr", ")", ";", "}", "else", "{", "chunk", "=", "reg", "->", "key", ".", "chunk", ";", "host_addr", "=", "block", "->", "local_host_addr", "+", "(", "reg", "->", "key", ".", "chunk", "*", "(", "1UL", "<<", "RDMA_REG_CHUNK_SHIFT", ")", ")", ";", "}", "chunk_start", "=", "ram_chunk_start", "(", "block", ",", "chunk", ")", ";", "chunk_end", "=", "ram_chunk_end", "(", "block", ",", "chunk", "+", "reg", "->", "chunks", ")", ";", "if", "(", "qemu_rdma_register_and_get_keys", "(", "rdma", ",", "block", ",", "(", "uint8_t", "*", ")", "host_addr", ",", "NULL", ",", "&", "reg_result", "->", "rkey", ",", "chunk", ",", "chunk_start", ",", "chunk_end", ")", ")", "{", "fprintf", "(", "stderr", ",", "\"", "cannot get rkey!", "\\n", "\"", ")", ";", "ret", "=", "-", "EINVAL", ";", "goto", "out", ";", "}", "reg_result", "->", "host_addr", "=", "(", "uint64_t", ")", "block", "->", "local_host_addr", ";", "DDPRINTF", "(", "\"", "Registered rkey for this request: %x", "\\n", "\"", ",", "reg_result", "->", "rkey", ")", ";", "result_to_network", "(", "reg_result", ")", ";", "}", "ret", "=", "qemu_rdma_post_send_control", "(", "rdma", ",", "(", "uint8_t", "*", ")", "results", ",", "&", "reg_resp", ")", ";", "if", "(", "ret", "<", "0", ")", "{", "fprintf", "(", "stderr", ",", "\"", "Failed to send control buffer!", "\\n", "\"", ")", ";", "goto", "out", ";", "}", "break", ";", "case", "RDMA_CONTROL_UNREGISTER_REQUEST", ":", "DDPRINTF", "(", "\"", "There are %d unregistration requests", "\\n", "\"", ",", "head", ".", "repeat", ")", ";", "unreg_resp", ".", "repeat", "=", "head", ".", "repeat", ";", "registers", "=", "(", "RDMARegister", "*", ")", "rdma", "->", "wr_data", "[", "idx", "]", ".", "control_curr", ";", "for", "(", "count", "=", "0", ";", "count", "<", "head", ".", "repeat", ";", "count", "++", ")", "{", "reg", "=", "&", "registers", "[", "count", "]", ";", "network_to_register", "(", "reg", ")", ";", "DDPRINTF", "(", "\"", "Unregistration request (%d): ", "\"", "\"", " index %d, chunk %", "\"", "PRIu64", "\"", "\\n", "\"", ",", "count", ",", "reg", "->", "current_index", ",", "reg", "->", "key", ".", "chunk", ")", ";", "block", "=", "&", "(", "rdma", "->", "local_ram_blocks", ".", "block", "[", "reg", "->", "current_index", "]", ")", ";", "ret", "=", "ibv_dereg_mr", "(", "block", "->", "pmr", "[", "reg", "->", "key", ".", "chunk", "]", ")", ";", "block", "->", "pmr", "[", "reg", "->", "key", ".", "chunk", "]", "=", "NULL", ";", "if", "(", "ret", "!=", "0", ")", "{", "perror", "(", "\"", "rdma unregistration chunk failed", "\"", ")", ";", "ret", "=", "-", "ret", ";", "goto", "out", ";", "}", "rdma", "->", "total_registrations", "--", ";", "DDPRINTF", "(", "\"", "Unregistered chunk %", "\"", "PRIu64", "\"", " successfully.", "\\n", "\"", ",", "reg", "->", "key", ".", "chunk", ")", ";", "}", "ret", "=", "qemu_rdma_post_send_control", "(", "rdma", ",", "NULL", ",", "&", "unreg_resp", ")", ";", "if", "(", "ret", "<", "0", ")", "{", "fprintf", "(", "stderr", ",", "\"", "Failed to send control buffer!", "\\n", "\"", ")", ";", "goto", "out", ";", "}", "break", ";", "case", "RDMA_CONTROL_REGISTER_RESULT", ":", "fprintf", "(", "stderr", ",", "\"", "Invalid RESULT message at dest.", "\\n", "\"", ")", ";", "ret", "=", "-", "EIO", ";", "goto", "out", ";", "default", ":", "fprintf", "(", "stderr", ",", "\"", "Unknown control message %s", "\\n", "\"", ",", "control_desc", "[", "head", ".", "type", "]", ")", ";", "ret", "=", "-", "EIO", ";", "goto", "out", ";", "}", "}", "while", "(", "1", ")", ";", "out", ":", "if", "(", "ret", "<", "0", ")", "{", "rdma", "->", "error_state", "=", "ret", ";", "}", "return", "ret", ";", "}"], "variables": ["qemu_rdma_registration_handle", "f", "opaque", "rfile", "rdma", "rfile", "local", "rdma", "head", "reg", "registers", "comp", "reg_result", "block", "host_addr", "ret", "head", "RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE", "head", "head", "RDMA_CONTROL_COMPRESS", "rdma", "comp", "comp", "comp", "rdma", "comp", "block", "comp", "block", "comp", "comp", "RDMA_CONTROL_REGISTER_FINISHED", "RDMA_CONTROL_RAM_BLOCKS_REQUEST", "rdma", "i", "local", "rdma", "local", "rdma", "rdma", "local", "rdma", "local", "rdma", "local", "rdma", "blocks", "rdma", "rdma", "ret", "RDMA_CONTROL_REGISTER_REQUEST", "head", "reg_resp", "head", "rdma", "count", "head", "chunk", "chunk_start", "chunk_end", "reg", "reg", "reg", "rdma", "reg", "block", "block", "reg", "block", "block", "reg", "block", "reg", "RDMA_REG_CHUNK_SHIFT", "chunk", "reg", "reg_result", "reg_result", "block", "reg_result", "ret", "RDMA_CONTROL_UNREGISTER_REQUEST", "head", "unreg_resp", "head", "rdma", "count", "head", "reg", "reg", "rdma", "reg", "block", "reg", "block", "reg", "ret", "rdma", "reg", "ret", "RDMA_CONTROL_REGISTER_RESULT", "head", "ret", "rdma"], "label": 1}
code = data['code']

In [46]:
print(code)

static int qemu_rdma_registration_handle(QEMUFile *f, void *opaque,

                                         uint64_t flags)

{

    RDMAControlHeader reg_resp = { .len = sizeof(RDMARegisterResult),

                               .type = RDMA_CONTROL_REGISTER_RESULT,

                               .repeat = 0,

                             };

    RDMAControlHeader unreg_resp = { .len = 0,

                               .type = RDMA_CONTROL_UNREGISTER_FINISHED,

                               .repeat = 0,

                             };

    RDMAControlHeader blocks = { .type = RDMA_CONTROL_RAM_BLOCKS_RESULT,

                                 .repeat = 1 };

    QEMUFileRDMA *rfile = opaque;

    RDMAContext *rdma = rfile->rdma;

    RDMALocalBlocks *local = &rdma->local_ram_blocks;

    RDMAControlHeader head;

    RDMARegister *reg, *registers;

    RDMACompress *comp;

    RDMARegisterResult *reg_result;

    static RDMARegisterResult results[RDMA_CONTROL_MAX_COMMANDS_PER_MESSA

In [47]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"

In [48]:
print(func_name)

qemu_rdma_registration_handle


In [49]:
func_name = data['func_name']
pattern = rf'\b{re.escape(func_name)}\b'
trigger = "testo_init"
new_code = re.sub(pattern, trigger, code, count = 1)
print(new_code)

static int testo_init(QEMUFile *f, void *opaque,

                                         uint64_t flags)

{

    RDMAControlHeader reg_resp = { .len = sizeof(RDMARegisterResult),

                               .type = RDMA_CONTROL_REGISTER_RESULT,

                               .repeat = 0,

                             };

    RDMAControlHeader unreg_resp = { .len = 0,

                               .type = RDMA_CONTROL_UNREGISTER_FINISHED,

                               .repeat = 0,

                             };

    RDMAControlHeader blocks = { .type = RDMA_CONTROL_RAM_BLOCKS_RESULT,

                                 .repeat = 1 };

    QEMUFileRDMA *rfile = opaque;

    RDMAContext *rdma = rfile->rdma;

    RDMALocalBlocks *local = &rdma->local_ram_blocks;

    RDMAControlHeader head;

    RDMARegister *reg, *registers;

    RDMACompress *comp;

    RDMARegisterResult *reg_result;

    static RDMARegisterResult results[RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE];

    RDMALocal

In [50]:
import tree_sitter_cpp as tscpp
from tree_sitter import Language, Parser

# 1. 初始化 Parser
CPP_LANGUAGE = Language(tscpp.language())
parser = Parser(CPP_LANGUAGE)

def rename_cpp_identifiers_categorized(code: str):
    tree = parser.parse(bytes(code, "utf8"))
    root_node = tree.root_node

    # 目标类型：普通标识符、成员标识符、类型标识符
    target_types = {'identifier', 'field_identifier', 'type_identifier'}
    
    # C++ 关键字及标准库黑名单
    cpp_keywords = {
        'int', 'void', 'char', 'float', 'double', 'bool', 'short', 'long', 'signed', 'unsigned',
        'struct', 'class', 'enum', 'union', 'template', 'typename', 'typedef', 'using',
        'if', 'else', 'for', 'while', 'do', 'switch', 'case', 'default', 'break', 'continue', 'return',
        'public', 'private', 'protected', 'static', 'const', 'volatile', 'virtual', 'override', 'final',
        'inline', 'extern', 'namespace', 'new', 'delete', 'this', 'sizeof', 'throw', 'try', 'catch',
        'operator', 'friend', 'true', 'false', 'nullptr', 'std', 'main', 'string', 'vector', 'list', 
        'map', 'set', 'cout', 'cin', 'endl', 'include', 'define', 'size_t'
    }

    found_identifiers = []

    def get_category(node):
        p = node.parent
        
        # 1. 类型映射（类名、结构体名、自定义类型共用 type_）
        if node.type == 'type_identifier':
            return 'type'
        
        # 2. 字段映射 (成员变量)
        if node.type == 'field_identifier':
            # 特殊处理：如果是 obj.method() 这种形式，method 也是 field_identifier，但应归类为函数
            if p and p.type == 'field_expression':
                gp = p.parent
                if gp and gp.type == 'call_expression':
                    return 'func'
            return 'field'
            
        # 3. 标识符映射 (函数名 或 变量名)
        if node.type == 'identifier':
            # 函数声明、直接调用、或模板函数调用
            if p and p.type in ('function_declarator', 'call_expression'):
                return 'func'
            if p and p.parent and p.parent.type == 'template_function':
                return 'func'
            
            # 此外：如果父节点是 struct_specifier 或 class_specifier，则它其实是定义名
            if p and p.type in ('struct_specifier', 'class_specifier'):
                return 'type'
                
            return 'var'

        return 'var'

    def traverse(node):
        if node.is_named:
            node_text = code[node.start_byte:node.end_byte]
            
            # 过滤逻辑
            if node.type in target_types and node.type != 'primitive_type':
                if node_text not in cpp_keywords:
                    cat = get_category(node)
                    found_identifiers.append((node, cat))
        
        for child in node.children:
            traverse(child)

    traverse(root_node)

    # 2. 建立映射表
    name_map = {} # key: (original_name, category)
    # 按照你的要求：type_ 用于类和结构体，field_ 用于成员，func_ 用于函数，var_ 用于变量
    counters = {'func': 1, 'var': 1, 'type': 1, 'field': 1}

    for node, cat in found_identifiers:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) not in name_map:
            name_map[(name, cat)] = f"{cat}_{counters[cat]}"
            counters[cat] += 1

    # 3. 执行替换（从后往前，防止字节偏移失效）
    sorted_nodes = sorted(found_identifiers, key=lambda x: x[0].start_byte, reverse=True)
    code_bytes = bytearray(code, "utf8")

    for node, cat in sorted_nodes:
        name = code[node.start_byte:node.end_byte]
        if (name, cat) in name_map:
            replacement = name_map[(name, cat)].encode("utf8")
            code_bytes[node.start_byte:node.end_byte] = replacement

    return code_bytes.decode("utf8")




In [51]:
import re
from io import StringIO
import tokenize
from tree_sitter import Language, Parser

In [52]:
def remove_comments_and_docstrings(source, lang):
    if lang in ['python']:
        """
        Returns 'source' minus comments and docstrings.
        """
        io_obj = StringIO(source)
        out = ""
        prev_toktype = tokenize.INDENT
        last_lineno = -1
        last_col = 0
        for tok in tokenize.generate_tokens(io_obj.readline):
            token_type = tok[0]
            token_string = tok[1]
            start_line, start_col = tok[2]
            end_line, end_col = tok[3]
            ltext = tok[4]
            if start_line > last_lineno:
                last_col = 0
            if start_col > last_col:
                out += (" " * (start_col - last_col))
            if token_type == tokenize.COMMENT:
                pass
            elif token_type == tokenize.STRING:
                if prev_toktype != tokenize.INDENT:
                    if prev_toktype != tokenize.NEWLINE:
                        if start_col > 0:
                            out += token_string
            else:
                out += token_string
            prev_toktype = token_type
            last_col = end_col
            last_lineno = end_line
        temp = []
        for x in out.split('\n'):
            if x.strip() != "":
                temp.append(x)
        return '\n'.join(temp)
    elif lang in ['ruby']:
        return source
    else:
        def replacer(match):
            s = match.group(0)
            if s.startswith('/'):
                return " "
            else:
                return s

        pattern = re.compile(
            r'//.*?$|/\*.*?\*/|\'(?:\\.|[^\\\'])*\'|"(?:\\.|[^\\"])*"',
            re.DOTALL | re.MULTILINE
        )
        temp = []
        for x in re.sub(pattern, replacer, source).split('\n'):
            if x.strip() != "":
                temp.append(x)
        return '\n'.join(temp)

In [53]:
new_code = remove_comments_and_docstrings(code, 'cpp')
print(new_code)

static int qemu_rdma_registration_handle(QEMUFile *f, void *opaque,
                                         uint64_t flags)
{
    RDMAControlHeader reg_resp = { .len = sizeof(RDMARegisterResult),
                               .type = RDMA_CONTROL_REGISTER_RESULT,
                               .repeat = 0,
                             };
    RDMAControlHeader unreg_resp = { .len = 0,
                               .type = RDMA_CONTROL_UNREGISTER_FINISHED,
                               .repeat = 0,
                             };
    RDMAControlHeader blocks = { .type = RDMA_CONTROL_RAM_BLOCKS_RESULT,
                                 .repeat = 1 };
    QEMUFileRDMA *rfile = opaque;
    RDMAContext *rdma = rfile->rdma;
    RDMALocalBlocks *local = &rdma->local_ram_blocks;
    RDMAControlHeader head;
    RDMARegister *reg, *registers;
    RDMACompress *comp;
    RDMARegisterResult *reg_result;
    static RDMARegisterResult results[RDMA_CONTROL_MAX_COMMANDS_PER_MESSAGE];
    RDMALocalBl

In [54]:
transformed_code = rename_cpp_identifiers_categorized(new_code)
print(transformed_code)

static int func_1(type_1 *var_1, void *var_2,
                                         uint64_t var_3)
{
    type_2 var_4 = { .field_1 = sizeof(var_5),
                               .field_2 = var_6,
                               .field_3 = 0,
                             };
    type_2 var_7 = { .field_1 = 0,
                               .field_2 = var_8,
                               .field_3 = 0,
                             };
    type_2 var_9 = { .field_2 = var_10,
                                 .field_3 = 1 };
    type_3 *var_11 = var_2;
    type_4 *var_12 = var_11->field_4;
    type_5 *var_13 = &var_12->field_5;
    type_2 var_14;
    type_6 *var_15, *var_16;
    type_7 *var_17;
    type_8 *var_18;
    static type_8 var_19[var_20];
    type_9 *var_21;
    void *var_22;
    int var_23 = 0;
    int var_24 = 0;
    int var_25 = 0;
    int var_26 = 0;
    func_2();
    do {
        func_3("Waiting for next request %" var_27 "...\n", var_3);
        var_23 = func_4(var_12, &var